[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Surjasa/teen-mental-health-analysis/blob/main/notebooks/03_descriptive_analysis_visualization.ipynb)

# 03 Descriptive Analysis and Visualization

This notebook summarizes the raw dataset, inspect distributions, and generate paper-ready figures before statistical inference.

The first code cell imports the plotting and data libraries:

- `pandas` for reading and handling the CSV
- `matplotlib.pyplot` for plotting
- `seaborn` for statistical visualizations

It also sets Seaborn’s theme to `whitegrid`, which makes the plots cleaner and easier to read.

So this cell is just the setup cell. It does not analyze the data yet; it prepares the notebook for the descriptive analysis cells that come after it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

**Quick descriptive summary of the raw dataset**

- `df.shape` → how many rows and columns you have.
- `df.head()` → a preview of the actual records.
- `df.describe(include='all').T` → a summary table for every column.

What that summary table means:
- `count` = how many non-missing values.
- `unique` = number of distinct values in a categorical column.
- `top` = most frequent category.
- `freq` = how often that top category appears.
- `mean` = average for numeric columns.
- `std` = standard deviation.
- `min`, `25%`, `50%`, `75%`, `max` = spread of numeric values.

Why this is useful for the paper:
- Provides the dataset overview to include in the Methods/Data section.
- Lets us report sample size (e.g., 1200 samples, 13 variables).
- Reveals class imbalance (e.g., `depression_label`), variable spreads, and types (numeric vs categorical).

Why it belongs in this notebook:
- This is the **descriptive analysis and visualization** notebook; its job is to summarize the raw dataset before inference.
- These outputs guide which figures, tables, and analyses to include in the paper (e.g., distributions, correlations, counts).
- It is intentionally separate from preprocessing because here we are *describing* the data, not transforming it for modeling.

What we can extract for the paper:
- Dataset overview (rows × columns)
- Variable types and simple statistics
- Notes on class imbalance and any notable distributions

Run the cell after this one to generate the actual preview (`df.head()`) and the summary table (`df.describe(include='all').T`).

Distribution of demographic and clinical variables in the Teen Mental Health Dataset (N=1200).

In [ ]:
from google.colab import files
print('Select dataset (CSV) to upload when prompted.')
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
print('Shape:', df.shape)
display(df.head())
display(df.describe(include='all').T)

In [ ]:
target_col = 'depression_label'
numeric_cols = df.select_dtypes(include='number').columns.difference([target_col]).tolist()
categorical_cols = df.select_dtypes(exclude='number').columns.difference([target_col]).tolist()

# quick checks
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)
print(df[numeric_cols].dtypes)

For this dataset, keep `academic_performance`, `daily_social_media_hours`, `screen_time_before_sleep`, and `sleep_hours` numeric; treat `age`, `addiction_level`, `anxiety_level`, `stress_level`, and `physical_activity` as discrete or ordinal-style variables for interpretation; and keep `gender`, `platform_usage`, and `social_interaction_level` categorical.

In [ ]:
# inspect
print(df[numeric_cols].dtypes)
print(df[numeric_cols].isna().sum())
print(df[categorical_cols].isna().sum())

for c in numeric_cols:
    print(c, "n_unique =", df[c].nunique(), "range =", df[c].min(), "-", df[c].max())

In [ ]:
# dtypes and missing
print('\nNumeric dtypes:')
print(df[numeric_cols].dtypes)
print('\nMissing in numeric cols:')
print(df[numeric_cols].isna().sum())
print('\nMissing in categorical cols:')
print(df[categorical_cols].isna().sum())

In [ ]:
import numpy as np
import pandas as pd

# Detect object columns that are actually numeric strings and coerce them safely
obj_cols = df.select_dtypes(include="object").columns.tolist()

converted_cols = []
for c in obj_cols:
    coerced = pd.to_numeric(
        df[c].astype(str).str.replace(",", "").str.strip(), errors="coerce"
    )
    n_coercible = coerced.notna().sum()

    # If more than 90% of the column can be cleanly converted to numbers, commit it
    if len(df) > 0 and (n_coercible / len(df) > 0.9):
        df[c] = coerced
        converted_cols.append((c, n_coercible, len(df)))
        print(
            f"Column '{c}' looks numeric-like — converted to numeric (coerced {n_coercible}/{len(df)})"
        )
print("\n--- Current Data Type Distribution ---")
print(df.dtypes.value_counts())
print("\n--- Descriptive Summary Preview (Numeric Features) ---")
# Generates a quick, clear statistical overview of the numerical data
display(df.describe().T.round(2))

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6, 3))
    series = df[col].dropna()
    use_kde = series.nunique() > 20
    sns.histplot(series, bins=20, kde=use_kde, color='steelblue')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

Improved categorical plots: shows counts with percentages, orders bars by frequency, switches to horizontal bars when a variable has many categories, and includes an example to compare categories by the target.

In [ ]:
for col in categorical_cols:
    counts = df[col].dropna().value_counts()
    total = counts.sum()
    order = counts.index

    # choose horizontal layout when many categories
    plt.figure(figsize=(10, 5) if len(order) > 8 else (6, 3))
    if len(order) > 8:
        sns.barplot(x=counts.values, y=order, color='steelblue')
        plt.xlabel('Count')
        plt.ylabel(col)
        for i, v in enumerate(counts.values):
            plt.text(v + total * 0.01, i, f"{v} ({v/total:.0%})", va='center', fontsize=8)
    else:
        sns.barplot(x=order, y=counts.values, color='steelblue', order=order)
        plt.xlabel(col)
        plt.ylabel('Count')
        for i, v in enumerate(counts.values):
            plt.text(i, v + total * 0.01, f"{v} ({v/total:.0%})", ha='center', va='bottom', fontsize=8)
        plt.xticks(rotation=20)

    plt.title(f'Counts of {col}')
    plt.tight_layout()
    plt.show()



In [ ]:
# Categorical features vs target: grouped barplots + contingency tables
import numpy as np
from pathlib import Path

reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    # ordered categories by overall frequency
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue='depression_label', order=order)
    plt.title(f'{col} by depression_label')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    # print contingency table and save
    ct = pd.crosstab(df[col], df['depression_label'])
    display(ct)
    ct.to_csv(reports_dir / f'ct_{col}_by_depression.csv')


In [ ]:
plt.figure(figsize=(10, 8))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Descriptive analysis and visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)

# Missingness summary: always save CSV; plot only if any missing values exist
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
missing_pct_df = missing_pct.reset_index().rename(columns={'index':'column', 0:'percent_missing'})
missing_pct_df.to_csv(reports_dir / 'missing_pct_by_column.csv', index=False)
if missing_pct.max() == 0:
    print('No missing values detected in the dataset.')
    (reports_dir / 'no_missing.txt').write_text('No missing values detected: all columns complete.')
else:
    plt.figure(figsize=(10,4))
    sns.barplot(x=missing_pct.index, y=missing_pct.values, color='steelblue')
    plt.ylabel('% missing')
    plt.xticks(rotation=45, ha='right')
    plt.title('Percent missing by column')
    plt.tight_layout()
    plt.savefig(reports_dir / 'missing_values_percent.png', dpi=150, bbox_inches='tight')
    plt.show()

# Summary statistics
print('Summary statistics:')
print(df.describe(include='all').T)

# Numeric distributions
num_cols = df.select_dtypes(include=['number']).columns.tolist()
for c in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[c].dropna(), kde=True)
    plt.title(f'Distribution: {c}')
    plt.savefig(reports_dir / f'fig_{c}.png', dpi=150, bbox_inches='tight')
    plt.close()

# Categorical counts
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for c in cat_cols:
    plt.figure(figsize=(6,4))
    df[c].value_counts().plot(kind='bar')
    plt.title(f'Counts: {c}')
    plt.savefig(reports_dir / f'fig_{c}_counts.png', dpi=150, bbox_inches='tight')
    plt.close()

# Correlation heatmap for numeric features
if len(num_cols) > 1:
    plt.figure(figsize=(10,8))
    sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Numeric feature correlations')
    plt.savefig(reports_dir / 'corr_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

print(f'All figures saved to {reports_dir.resolve()}')


**Note about the missingness sentinel:**

When the dataset contains no missing values, this notebook writes a small sentinel file `reports/no_missing.txt` to indicate completeness. If that file is present, no percent-missing plot is generated; otherwise the notebook will save `reports/missing_values_percent.png` showing the percent-missing for each column.

In [ ]:
# Pairwise plots for top-6 features by association with the target (simple, uncluttered views)
from itertools import combinations
from scipy.stats import chi2_contingency

reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)

target = 'depression_label'
# Recompute feature lists in case notebook state changed
numeric_cols = df.select_dtypes(include='number').columns.difference([target]).tolist()
categorical_cols = df.select_dtypes(exclude='number').columns.difference([target]).tolist()
#Extra-ish (it is a statistical association measure used for ranking, not a plot.)
# helper: Cramer's V
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.values.sum()
    phi2 = chi2 / n if n > 0 else 0
    r, k = confusion_matrix.shape
    denom = min(k - 1, r - 1)
    return (phi2 / denom) ** 0.5 if denom > 0 else 0

#Extra analysis: the feature-ranking part before plotting
# association scores
assoc_scores = {}
for c in numeric_cols:
    try:
        assoc_scores[c] = abs(df[c].corr(df[target]))
    except Exception:
        assoc_scores[c] = 0
for c in categorical_cols:
    ct = pd.crosstab(df[c], df[target])
    assoc_scores[c] = cramers_v(ct) if ct.size else 0

assoc_series = pd.Series(assoc_scores).sort_values(ascending=False)
top6 = assoc_series.index[:6].tolist()
print('Top-6 features by association with target:', top6)

# Create one simple plot per pair and save
for a, b in combinations(top6, 2):
    plt.figure(figsize=(6,4))
    if a in numeric_cols and b in numeric_cols:
        sns.scatterplot(data=df, x=a, y=b, hue=target, alpha=0.55, s=35)
    elif (a in numeric_cols and b in categorical_cols) or (a in categorical_cols and b in numeric_cols):
        num = a if a in numeric_cols else b
        cat = b if b in categorical_cols else a
        sns.boxplot(data=df, x=cat, y=num)
    else:
        # categorical vs categorical: simple stacked bar chart of proportions
        ct = pd.crosstab(df[a], df[b], normalize='index')
        ax = ct.plot(kind='bar', stacked=True, figsize=(6,4), colormap='Pastel1', ax=plt.gca())
        plt.ylabel('Proportion')
        plt.xlabel(a)
        plt.legend(title=b, bbox_to_anchor=(1.05, 1), loc='upper left')
        # annotate percent labels inside sizable segments
        for i, (idx, row) in enumerate(ct.iterrows()):
            cum = 0
            for j, val in enumerate(row):
                if val > 0.02:  # skip very small segments
                    ax.text(i, cum + val/2, f'{val:.0%}', ha='center', va='center', fontsize=8)
                cum += val

    plt.title(f'{a} vs {b}')
    plt.tight_layout()
    savepath = reports_dir / f'pair_{a}__{b}.png'
    plt.savefig(savepath, dpi=150, bbox_inches='tight')
    plt.show()

# Pairplot for numeric subset of top6 using histograms on the diagonal (no KDE)
num_top = [c for c in top6 if c in numeric_cols]
if len(num_top) > 1:
    try:
        pp = sns.pairplot(df[num_top + [target]].dropna(), hue=target, diag_kind='hist', corner=True, plot_kws={'alpha': 0.6, 's': 25})
        pp.fig.suptitle('Pairwise numeric plots among top features', y=1.02)
        pp.savefig(reports_dir / 'pairplot_top_numeric.png', dpi=150, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print('pairplot failed:', e)

print(f'Pairwise figures saved to {reports_dir.resolve()}')
